In [2]:
# ruff: noqa: F401
import os
import random
from typing import Annotated

import gradio as gr
import requests
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from rich.console import Console
from rich.markdown import Markdown
from typing_extensions import TypedDict

load_dotenv(override=True)
console = Console()

llm = ChatOpenAI(model="gpt-5.4-nano")

In [3]:
# Define state for the graph
class State(TypedDict):
    messages: Annotated[list, add_messages]
    bengali: str


# Define chatbot node function
def chatbot_node_with_tools(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


# Define the translattor node function
def translator_node(state: State) -> dict:
    text = state["messages"][-1].content
    prompt = f"Translate this into Bengali, replying with translation only: \n\n{text}"
    return {"bengali": llm.invoke(prompt).content}


# Define the tools


@tool
def serper_search(query: str) -> str:
    """Search Google using the Serper API."""
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, json={"q": query})
    return response.text


@tool
def send_push_notifications(text: str) -> str:
    """Send push notification to mobile"""
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_TOKEN"),
            "user": os.getenv("PUSHOVER_USER"),
            "message": text,
        },
    )
    return "Notification sent"


tools = [serper_search, send_push_notifications]
llm_with_tools = llm.bind_tools(tools)

# Define the builder object
builder = StateGraph(State)

builder.add_node("chatbot", chatbot_node_with_tools)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)

# Define the edges
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("tools", "chatbot")
builder.add_edge("translator", END)

# Define the graph with memeory
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


config = {"configurable": {"thread_id": "conversation-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi my name is Anirban Guha"}]}, config)
second_result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]}, config
)
console.print(Markdown(second_result["messages"][-1].content))
console.log(Markdown(second_result["bengali"]))

Your name is Anirban Guha.

[21:42:56] আপনার নাম অনির্বাণ গুহা।                                                                        ]8;id=9215762;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_4431/256807608.py\256807608.py]8;;\:]8;id=9215763;file:///var/folders/mq/h979j9sj6f38h5m8d5wmkysw0000gp/T/ipykernel_4431/256807608.py#75\75]8;;\

### Persistent Memory with SQLite


In [4]:
with SqliteSaver.from_conn_string("memory.db") as sql_memory:
    durable_graph = builder.compile(checkpointer=memory)
    sql_config = {"configurable": {"thread_id": "conversation - 2"}}
    durable_graph.invoke(
        input={"messages": [{"role": "user", "content": "My favorite sports in soccer"}]},
        config=sql_config,
    )
    reply = graph.invoke(
        input={"messages": [{"role": "user", "content": "What is my favorite sports?"}]},
        config=sql_config,
    )

    console.print(Markdown(reply["messages"][-1].content))
    console.print(Markdown(reply["bengali"]))

I can’t know your favorite sports from what you’ve said so far.                                                    

If you tell me which sport(s) you like, I can help you describe them or suggest something similar.                 

So—what are your favorite sports? (Just list a couple.)

এখন পর্যন্ত আপনি যা বলেছেন, তা থেকে আমি আপনার পছন্দের খেলাগুলো জানতে পারছি না।                                                           

আপনি কোন খেলাগুলো পছন্দ করেন বললে, আমি সেগুলো বর্ণনা করতে সাহায্য করতে পারি বা একই রকম কিছু পরামর্শ দিতে পারি।                                    

তাই—আপনার প্রিয় খেলাগুলো কী? (শুধু দু’একটা তালিকা দিন।)

### Looking inside the state history


In [5]:
snapshot = graph.get_state(config)
messages = snapshot.values["messages"]
print("Messages saved: ", len(messages))

for message in messages:
    console.print(message.content)


Messages saved:  4


Hi my name is Anirban Guha

Hi Anirban Guha! Nice to meet you. How can I help you today?

What is my name?

Your name is **Anirban Guha**.

In [6]:
history = list(graph.get_state_history(config))
for h in reversed(history):
    m = h.metadata
    queue = ", ".join(t.name for t in h.tasks) or "(run complete)"
    print(
        f"Step {m['step']:>2}  {m['source']:<5} messages = "
        f"{len(h.values.get('messages', []))} about to run {queue}"
    )

Step -1  input messages = 0 about to run __start__
Step  0  loop  messages = 1 about to run chatbot
Step  1  loop  messages = 2 about to run translator
Step  2  loop  messages = 2 about to run (run complete)
Step  3  input messages = 2 about to run __start__
Step  4  loop  messages = 3 about to run chatbot
Step  5  loop  messages = 4 about to run translator
Step  6  loop  messages = 4 about to run (run complete)


In [7]:
# Taking a checkpoint and going back in history
earlier = history[3]
replay_config = {
    "configurable": {
        "thread_id": "conversation-1",
        "checkpoint_id": earlier.config["configurable"]["checkpoint_id"],
    }
}
resumed = graph.invoke(None, replay_config)
console.print(f"Resumed from past. The graph now holds {len(resumed['messages'])}, messages")

Resumed from past. The graph now holds 4, messages

#### Adding Gradio UI


In [12]:
def chat(user_input: str, history):
    config = {"configurable": {"thread_id": "gradio_chat"}}
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config,
    )
    return f"{result['messages'][-1].content}\n\n{result['bengali']}"


gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
